In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

## 1. 데이터 불러오기

In [2]:
dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name="target")  # 0: 악성, 1: 양성

print("X 형태 (샘플 수, 특성 수):", X.shape)
print("클래스 비율:")
print(y.value_counts(normalize=True))

X 형태 (샘플 수, 특성 수): (569, 30)
클래스 비율:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64


## 2. 데이터 분할(계층 분할 + 재현성 고정)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
)

print("훈련 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)

훈련 데이터: (455, 30)
테스트 데이터: (114, 30)


## 3 ~ 6 트리 깊이별 과적합 확인

In [4]:
depth_list = [1, 2, 3, 5, 10, None]   # 실험할 깊이 6개 (None = 제한 없음)
results = []                          # 깊이별 결과를 모을 빈 리스트

for depth in depth_list:              # 깊이를 하나씩 꺼내서 반복
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)  # 이번 깊이로 빈 모델 생성

    # 학습
    model.fit(X_train, y_train)       # train으로 규칙 학습

    # 훈련 점수 (연습문제 점수)
    train_pred = model.predict(X_train)            # 배운 데이터를 다시 예측
    train_acc = accuracy_score(y_train, train_pred) # 정답과 비교해 정답률 계산

    # 교차검증 점수 (모의고사 5번 평균) - 훈련 데이터 안에서만 수행
    cv_scores = cross_val_score(      # train을 5조각으로 나눠 5번 학습·검증 → 점수 5개
        model,                        # 평가할 모델 (설정만 가져가서 매번 새로 학습)
        X_train,                      # train만 사용, test는 안 넣음
        y_train,
        cv=5,                         # 5조각
        scoring="accuracy"            # 정답률로 채점
    )
    cv_mean = cv_scores.mean()        # 점수 5개의 평균 → 예상 실력
    cv_std = cv_scores.std()          # 점수 5개의 표준편차 → 안정성

    results.append({                  # 이번 깊이 결과를 딕셔너리로 리스트에 추가
        "max_depth": str(depth),      # None이 NaN으로 바뀌지 않게 문자열로 저장
        "train_acc": round(train_acc, 3),          # 소수 셋째 자리까지
        "cv_mean": round(cv_mean, 3),
        "cv_std": round(cv_std, 3),
        "gap": round(train_acc - cv_mean, 3)       # train과 cv 차이 → 클수록 과적합
    })

result_df = pd.DataFrame(results)     # 딕셔너리 6개 → 6행짜리 표
print(result_df)                      # 결과표 출력


  max_depth  train_acc  cv_mean  cv_std    gap
0         1      0.923    0.897   0.019  0.026
1         2      0.958    0.919   0.019  0.040
2         3      0.976    0.925   0.024  0.051
3         5      0.993    0.932   0.021  0.062
4        10      1.000    0.910   0.019  0.090
5      None      1.000    0.910   0.019  0.090


## 7. 교차검증 기준으로 최적 깊이 선택

In [5]:
best_row = result_df.loc[result_df["cv_mean"].idxmax()]  # cv_mean 최대인 행 번호 찾기 → 그 행 꺼내기
best_depth_text = best_row["max_depth"]                   # 그 행의 깊이 (문자열, 예: "5")

if best_depth_text == "None":         # 문자열 "None"이면
    best_depth = None                 # 진짜 None으로 되돌림
else:
    best_depth = int(best_depth_text) # 아니면 "5" → 5 숫자로 변환

final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)  # 고른 깊이로 새 모델
final_model.fit(X_train, y_train)     # train 전체로 다시 학습 (교차검증 땐 4/5만 썼으니까)

test_pred = final_model.predict(X_test)         # 처음 보는 test 예측 (여기서 처음이자 마지막으로 test 사용)
test_acc = accuracy_score(y_test, test_pred)    # 최종 정답률

print("선택된 깊이:", best_depth)      # 5
print("최종 테스트 정확도:", round(test_acc, 3))  # 0.921

선택된 깊이: 5
최종 테스트 정확도: 0.921
